In [1]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)
import re


/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


pandas: 3.0.2  polars: 1.39.3


In [2]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- col_ends_with ---
FIX_COL_ENDS_WITH_DATA_PD = pd.DataFrame({"policy_id": [1], "claim_id": [2], "claim_amt": [10.0]})
FIX_COL_ENDS_WITH_DATA_PL = pl.from_pandas(FIX_COL_ENDS_WITH_DATA_PD)
FIX_COL_ENDS_WITH_DATA = FIX_COL_ENDS_WITH_DATA_PD
FIX_COL_ENDS_WITH_PAT = "_id"
FIX_COL_ENDS_WITH_KWARGS = {}

# --- col_matches ---
FIX_COL_MATCHES_DATA_PD = pd.DataFrame({"Claim_A": [1], "claim_b": [2], "premium": [3]})
FIX_COL_MATCHES_DATA_PL = pl.from_pandas(FIX_COL_MATCHES_DATA_PD)
FIX_COL_MATCHES_DATA = FIX_COL_MATCHES_DATA_PD
FIX_COL_MATCHES_KWARGS = {"case": False, "regex": True}
FIX_COL_MATCHES_PAT = "claim"

# --- col_starts_with ---
FIX_COL_STARTS_WITH_DATA_PD = pd.DataFrame({"pol_num": [1], "pol_date": [2], "claim_id": [3]})
FIX_COL_STARTS_WITH_DATA_PL = pl.from_pandas(FIX_COL_STARTS_WITH_DATA_PD)

print("✅ Fixtures loaded")


✅ Fixtures loaded


In [3]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_col_ends_with(data, pat, kwargs):
    return list(data.columns[data.columns.str.endswith(pat, **kwargs)])
    return None

def before_col_matches(data, kwargs, pat):
    return list(data.columns[data.columns.str.contains(pat, **kwargs)])
    return None

def before_col_starts_with():
    def col_starts_with(data: pd.DataFrame,
                        pat: str,
                        **kwargs):
        return list(data.columns[data.columns.str.startswith(pat, **kwargs)])
    return col_starts_with

In [4]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_col_ends_with(data, pat, kwargs):
    return [c for c in data.columns if c.endswith(pat)]

def gen_col_matches(data, kwargs, pat):
    import re

    return list(
        [
            c
            for c in data.columns
            if (
                re.search(pat, c, **kwargs)
                if kwargs.get("regex", True)
                else (
                    (pat in c)
                    if kwargs.get("case", True)
                    else (pat.lower() in c.lower())
                )
            )
        ]
    )

def gen_col_starts_with():

    def col_starts_with(data: pl.DataFrame,
                        pat: str,
                        **kwargs):
        return list([c for c in data.columns if c.startswith(pat, **kwargs)])
    return col_starts_with

In [5]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [6]:
# === Tests: col_starts_with ===

try:
    _r = gen_col_starts_with()
    print("✅ L1 smoke gen_col_starts_with: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_col_starts_with: {type(_e).__name__}: {_e}")

try:
    _rb = before_col_starts_with()
    print("✅ L1 smoke before_col_starts_with: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_col_starts_with: {type(_e).__name__}: {_e}")

try:
    _rb = before_col_starts_with()(FIX_COL_STARTS_WITH_DATA_PD, "pol")
    _rg = gen_col_starts_with()(FIX_COL_STARTS_WITH_DATA_PL, "pol")
    print("✅ L2 equivalence col_starts_with: MATCH" if _rb == _rg else f"❌ L2 equivalence col_starts_with: MISMATCH — before={_rb}, gen={_rg}")
except Exception as _e:
    print(f"❌ L2 equivalence col_starts_with: setup error — {type(_e).__name__}: {_e}")

try:
    _rb = before_col_starts_with()(FIX_COL_STARTS_WITH_DATA_PD, "missing")
    _rg = gen_col_starts_with()(FIX_COL_STARTS_WITH_DATA_PL, "missing")
    print("✅ L3 edge col_starts_with no match: MATCH" if _rb == _rg else f"❌ L3 edge col_starts_with no match: MISMATCH — before={_rb}, gen={_rg}")
except Exception as _e:
    print(f"❌ L3 edge col_starts_with: {type(_e).__name__}: {_e}")

try:
    _pd_empty_rows = pd.DataFrame(columns=["pol_num", "claim_id"])
    _pl_empty_rows = pl.DataFrame(schema={"pol_num": pl.Null, "claim_id": pl.Null})
    _rb = before_col_starts_with()(_pd_empty_rows, "pol")
    _rg = gen_col_starts_with()(_pl_empty_rows, "pol")
    print("✅ L3 edge col_starts_with empty rows: MATCH" if _rb == _rg == ["pol_num"] else f"❌ L3 edge col_starts_with empty rows: MISMATCH — before={_rb}, gen={_rg}")
except Exception as _e:
    print(f"❌ L3 edge col_starts_with empty rows: {type(_e).__name__}: {_e}")

try:
    _rb = before_col_starts_with()(FIX_COL_STARTS_WITH_DATA_PD, ("pol", "claim"))
    _rg = gen_col_starts_with()(FIX_COL_STARTS_WITH_DATA_PL, ("pol", "claim"))
    print("✅ L3 edge col_starts_with tuple prefix: MATCH" if _rb == _rg else f"❌ L3 edge col_starts_with tuple prefix: MISMATCH — before={_rb}, gen={_rg}")
except Exception as _e:
    print(f"❌ L3 edge col_starts_with tuple prefix: {type(_e).__name__}: {_e}")


✅ L1 smoke gen_col_starts_with: OK, type= function
✅ L1 smoke before_col_starts_with: OK
✅ L2 equivalence col_starts_with: MATCH
✅ L3 edge col_starts_with no match: MATCH
✅ L3 edge col_starts_with empty rows: MATCH
✅ L3 edge col_starts_with tuple prefix: MATCH
